# Leaf localisation — local training pipeline (VS Code)

Local, Colab-free version of `train_colab.ipynb`. Runs the same pipeline —
download the PlantDoc-YOLO dataset from Kaggle, collapse all 29
species/disease classes into one `leaf` class, train a YOLO26 single-class
leaf detector, validate it, and run it on a sample image — entirely on
your own machine, using this repo's existing `data.yaml` and `scripts/`.

**Prerequisites**
- Python env with the `ipykernel` the VS Code Jupyter extension is using
  selected as this notebook's kernel (top-right kernel picker).
- A [Kaggle](https://www.kaggle.com/) account (for downloading the dataset).
- A GPU is optional but strongly recommended — training 120 epochs at
  imgsz=960 on CPU is impractical. Without a local GPU, either run
  `train_colab.ipynb` on Colab's free T4 and copy `best.pt` back here, or
  set `RUN_TRAINING = False` further down to validate/infer with weights
  you already have.

## 1. Locate the repo root

The Jupyter kernel's working directory can start almost anywhere — the workspace root, the notebook's own folder, or (on a remote/JupyterHub kernel) your home directory — depending on `jupyter.notebookFileRoot` and how the kernel was launched. Resolve the repo root explicitly instead of assuming.

In [ ]:
import os
from pathlib import Path


def find_repo_root():
    # Search upward from every plausible starting point until data.yaml
    # turns up, so this works no matter where the notebook file actually
    # lives in the repo (root, notebooks/, or anywhere else).
    start_points = []

    # VS Code's Jupyter extension injects this global with the notebook's own
    # path, which works even when the kernel's cwd is unrelated (e.g. a
    # remote/JupyterHub kernel defaulting to your home directory).
    ipynb_path = globals().get("__vsc_ipynb_file__")
    if ipynb_path:
        start_points.append(Path(ipynb_path).resolve().parent)

    start_points.append(Path.cwd())

    for start in start_points:
        for candidate in [start, *start.parents]:
            if (candidate / "data.yaml").exists():
                return candidate

    return None


REPO_ROOT = find_repo_root()
assert REPO_ROOT is not None, (
    "Could not find data.yaml near the notebook's own location or the "
    f"kernel's working directory ({Path.cwd()}). Set REPO_ROOT manually to "
    "the leaf-localisation repo path, e.g.:\n"
    '  REPO_ROOT = Path("/path/to/leaf-localisation")'
)
os.chdir(REPO_ROOT)
print("Working directory set to:", Path.cwd())

## 2. Install packages

`%pip` (rather than `!pip`) installs into the kernel's own environment, which is what you want when the notebook's kernel isn't necessarily the same Python as your terminal's.

In [ ]:
%pip install -q ultralytics kaggle

## 3. Check for a GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"CUDA available — training will use GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU detected — training will fall back to CPU (slow). "
          "See the prerequisites note above.")

## 4. Kaggle credentials

Unlike Colab's file-upload widget, place `kaggle.json` directly in your home directory once and every project can reuse it.

In [ ]:
kaggle_json = Path.home() / ".kaggle" / "kaggle.json"

if kaggle_json.exists():
    print(f"Found Kaggle credentials at {kaggle_json}")
else:
    print(
        "Kaggle credentials not found.\n\n"
        "1. Go to https://www.kaggle.com/settings -> API -> Create New Token "
        "(downloads kaggle.json)\n"
        f"2. Move it to: {kaggle_json}\n"
        "3. On macOS/Linux, also run: chmod 600 ~/.kaggle/kaggle.json"
    )

## 5. Download the dataset

In [ ]:
!kaggle datasets download -d nirmalsankalana/plantdoc-yolo-26-dataset -p data/raw --unzip

## 6. Inspect the raw folder layout

Using `pathlib` instead of the shell's `find` keeps this cell working the same way on Windows, macOS, and Linux.

In [ ]:
raw_dir = Path("data/raw")
for p in sorted(raw_dir.rglob("*")):
    if p.is_dir() and len(p.relative_to(raw_dir).parts) <= 2:
        print(p)

In [ ]:
# Adjust this if the listing above shows train/val/test nested elsewhere
RAW_ROOT = Path("data/raw/yolo_dataset")

## 7. Relabel: collapse all 29 classes into one `leaf` class (id 0)

Reuses `relabel_file()` from `scripts/relabel.py` (copies images instead of symlinking, since Windows restricts symlink creation to admin users by default).

In [ ]:
import shutil
import sys

sys.path.insert(0, str(REPO_ROOT / "scripts"))
from relabel import relabel_file

PROCESSED = Path("data/processed")
SPLITS = ["train", "val", "test"]

for split in SPLITS:
    src_images = RAW_ROOT / split / "images"
    src_labels = RAW_ROOT / split / "labels"
    dst_images = PROCESSED / split / "images"
    dst_labels = PROCESSED / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    n_img = 0
    for src in src_images.iterdir():
        dst = dst_images / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
        n_img += 1

    n_lbl = 0
    for src in src_labels.iterdir():
        relabel_file(src, dst_labels / src.name)
        n_lbl += 1

    print(f"{split}: {n_img} images copied, {n_lbl} labels relabeled")

## 8. Sanity check

Reuses `scripts/verify.py` — checks every image has a matching label (and vice versa), counts boxes per split, and draws one sample image's boxes to `data/processed/sample_check.png`.

Expect: train images=1544 boxes=5280, val images=441 boxes=1593, test images=220 boxes=715, and zero missing labels/images. If these numbers don't match, stop and check `RAW_ROOT` in step 6 before training.

In [ ]:
%run scripts/verify.py

In [ ]:
from IPython.display import Image as IPImage, display

display(IPImage(filename="data/processed/sample_check.png", width=600))

## 9. Training config

`data.yaml` at the repo root already points `path:` at `data/processed`, so it's used as-is — no need to regenerate it like the Colab notebook does.

Set `RUN_TRAINING = False` to skip straight to validation/inference below using weights you already have (e.g. `best.pt` trained on Colab and copied into `runs/detect/train/weights/best.pt`).

In [ ]:
RUN_TRAINING = True

MODEL = "yolo26m.pt"
EPOCHS = 120
IMGSZ = 1280
BATCH = 6
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print(f"model={MODEL} epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} device={DEVICE}")

## 10. Train YOLO26

Uses the Ultralytics Python API directly (rather than the `yolo` CLI) so `DEVICE` and the other config variables above are passed in without shell quoting to worry about. Weights are saved to the Ultralytics default location, `runs/detect/train/weights/best.pt` — the same path `scripts/infer_area.py` and `scripts/webcam_demo.py` expect by default.

In [ ]:
from ultralytics import YOLO

if RUN_TRAINING:
    model = YOLO(MODEL)
    model.train(data="data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE)
else:
    print("RUN_TRAINING is False — skipping training.")

## 11. Validate

Picks up the weights training just produced, or falls back to the default local path if training was skipped.

In [ ]:
WEIGHTS = "runs/detect/train/weights/best.pt"

model = YOLO(WEIGHTS)
metrics = model.val(data="data.yaml")

print(f"Precision  : {metrics.box.mp:.3f}")
print(f"Recall     : {metrics.box.mr:.3f}")
print(f"mAP50      : {metrics.box.map50:.3f}")
print(f"mAP50-95   : {metrics.box.map:.3f}")

## 12. Try it on a sample image

No upload widget needed locally — just point `SAMPLE_IMAGE` at any image on disk (defaults to the first test image).

In [ ]:
CONF = 0.25  # lower this (e.g. 0.1) to see more/weaker boxes, raise it to see only confident ones

test_images = sorted((PROCESSED / "test" / "images").iterdir())
SAMPLE_IMAGE = test_images[0]

result = model.predict(str(SAMPLE_IMAGE), conf=CONF, verbose=False)[0]

print(f"{SAMPLE_IMAGE.name}: {len(result.boxes)} leaf(es) detected")
for i, box in enumerate(result.boxes):
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    conf = float(box.conf[0])
    area_px = (x2 - x1) * (y2 - y1)
    print(f"  #{i}: conf={conf:.2f}  box=({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})  area_px={area_px:.0f}")

from PIL import Image

display(Image.fromarray(result.plot()[..., ::-1]))  # plot() returns BGR; flip to RGB for display

## Next steps

- Batch inference over a whole folder, written out as a CSV of per-leaf areas: `python scripts/infer_area.py --source data/processed/test/images`
- Live webcam demo: `python scripts/webcam_demo.py`

Both default to `runs/detect/train/weights/best.pt`, so they'll pick up the weights trained above with no extra flags.